In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os, glob
import seaborn as sns
from matplotlib.patches import Rectangle

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
concept_df = pd.read_pickle("../../data/processed/sv_clip_scores.pkl")
concept_df

In [ ]:
concept_axis_dfs = {}
concepts = concept_df.columns.tolist()
concepts.remove('path')
len(concepts)

In [ ]:
concept_df = concept_df.sample(n=10000).reset_index(drop=True)
concept_df

In [ ]:
self_feature_rows = []
spatial_feature_rows = []
# self_files = glob.glob("../../data/features/Raw/*/*/RS/Mocov3VITB-self-*.h5")
# spatial_files = glob.glob("../../data/features/Raw/*/*/RS/Mocov3VITB-spatial-*.h5")
self_files = glob.glob("../../data/features/Raw/*/*/SV/Mocov3VITB-self-*.h5")
spatial_files = glob.glob("../../data/features/Raw/*/*/SV/Mocov3VITB-spatial-*.h5")
for f in self_files:
    f_df = pd.read_hdf(f, key='data')
    f_df['index'] = f_df['index'].str.replace('/home/huangyingjing/', '/data/')
    # f_df['index'] = f_df['index'].str.replace('/home/huangyingjing/', '../../data/')
    f_df_sample = f_df[f_df['index'].isin(concept_df['path'].tolist())].reset_index(drop=True)
    if not f_df_sample.empty:
        self_feature_rows.append(f_df_sample)
        
for f in spatial_files:
    f_df = pd.read_hdf(f, key='data')
    f_df['index'] = f_df['index'].str.replace('/home/huangyingjing/', '/data/')
    # f_df['index'] = f_df['index'].str.replace('/home/huangyingjing/', '../../data/')
    f_df_sample = f_df[f_df['index'].isin(concept_df['path'].tolist())].reset_index(drop=True)
    if not f_df_sample.empty:
        spatial_feature_rows.append(f_df_sample)

self_feature_df = pd.concat(self_feature_rows, axis=0).reset_index(drop=True)
spatial_feature_df = pd.concat(spatial_feature_rows, axis=0).reset_index(drop=True)
self_feature_df.shape, spatial_feature_df.shape

In [ ]:
concept_df = concept_df.set_index('path').loc[self_feature_df['index'].tolist()].reset_index()
concept_df.shape

In [ ]:
alphas = [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]

reg = make_pipeline(
    StandardScaler(), 
    RidgeCV(alphas=alphas, scoring='r2', cv=5) # 5-fold CV
)

In [ ]:
results = []
for concept in concepts:
    y = concept_df[concept].values
    X_self = self_feature_df.drop(columns=['index']).values
    X_spatial = spatial_feature_df.drop(columns=['index']).fillna(0).values
    
    reg.fit(X_self, y)
    r2_self = reg.score(X_self, y)
    
    reg.fit(X_spatial, y)
    r2_spatial = reg.score(X_spatial, y)
    
    print(f"Concept: {concept}, R2 Self: {r2_self:.4f}, R2 Spatial: {r2_spatial:.4f}")
    results.append({
        'concept': concept,
        'r2_self': r2_self,
        'r2_spatial': r2_spatial
    })
results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.to_csv('../../data/processed/fig/clip_concept_sv.csv', index=False)